# Problem Statement : Graph-Based Clustering using Minimum Spanning Tree (MST)

Syllabus mapping: Graph-based clustering, clustering with Minimum Spanning Tree
Objective: Implement MST-based clustering and compare it with a standard partitioning method.

Dataset (link):

Iris Dataset (UCI):

https://archive.ics.uci.edu/dataset/53/iris  

Problem Statement

You are building a clustering module for a data analytics toolkit. For a given dataset, you must perform graph-based clustering using an MST:

Build a complete weighted graph using pairwise distances
Construct the Minimum Spanning Tree
Form clusters by removing the (k−1) most expensive edges
Then compare this result with K-Means clustering.

Expected Input

iris.csv (features only; ignore labels for clustering)
Example:
python task2_mst_clustering.py --data iris.csv --k 3 --distance euclidean

Expected Output

Cluster labels for MST method:
mst_clusters.csv (SampleId, ClusterLabel)
Cluster labels for K-Means method:
kmeans_clusters.csv
Comparison report:
silhouette score for both methods
brief note: where MST clustering performs better/worse than K-Means
Mandatory Tasks / Deliverables

Implement MST via Kruskal/Prim (library allowed but document it)
Support at least one distance metric
Provide a short explanation (8–10 lines) of why MST can capture non-spherical clusters

In [16]:
import pandas as pd
import numpy as np

from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree, connected_components
from scipy.sparse import csr_matrix

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [17]:

data = pd.read_csv('IRIS.csv')

print("Dataset shape:", data.shape)
data.head()

Dataset shape: (150, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [18]:
X = data.iloc[:, :-1].values
print("Feature matrix shape:", X.shape)

Feature matrix shape: (150, 4)


In [19]:
distance_matrix = squareform(pdist(X, metric='euclidean'))

print("Distance matrix shape:", distance_matrix.shape)

Distance matrix shape: (150, 150)


In [20]:
mst_sparse = minimum_spanning_tree(distance_matrix)
mst = mst_sparse.toarray()

# convert to symmetric matrix
mst = mst + mst.T

mst[:5, :5]

array([[0.        , 0.        , 0.        , 0.        , 0.14142136],
       [0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.14142136, 0.        , 0.        , 0.        , 0.        ]])

In [21]:
k = 3
edges = []

for i in range(len(mst)):
    for j in range(i+1, len(mst)):
        if mst[i][j] > 0:
            edges.append((i, j, mst[i][j]))

# Sort edges descending by weight
edges.sort(key=lambda x: x[2], reverse=True)

# Remove largest (k-1) edges
for i in range(k-1):
    u, v, w = edges[i]
    mst[u][v] = 0
    mst[v][u] = 0

In [22]:
graph_sparse = csr_matrix(mst)

num_clusters, mst_labels = connected_components(graph_sparse)

print("Number of MST clusters:", num_clusters)
mst_labels[:10]

Number of MST clusters: 3


array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [23]:
mst_output = pd.DataFrame({
    "SampleId": range(len(mst_labels)),
    "ClusterLabel": mst_labels
})

mst_output.to_csv("mst_clusters.csv", index=False)

mst_output.head()

,SampleId,ClusterLabel
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


In [24]:
files.download("mst_clusters.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)

kmeans_labels = kmeans.fit_predict(X)

kmeans_labels[:10]

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=int32)

In [26]:
kmeans_output = pd.DataFrame({
    "SampleId": range(len(kmeans_labels)),
    "ClusterLabel": kmeans_labels
})

kmeans_output.to_csv("kmeans_clusters.csv", index=False)

files.download("kmeans_clusters.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
mst_score = silhouette_score(X, mst_labels)
kmeans_score = silhouette_score(X, kmeans_labels)

print("Silhouette Score (MST):", round(mst_score, 4))
print("Silhouette Score (KMeans):", round(kmeans_score, 4))

Silhouette Score (MST): 0.5118
Silhouette Score (KMeans): 0.5526


In [28]:
if mst_score > kmeans_score:
    print("MST clustering performs better because it captures irregular cluster structure.")
else:
    print("KMeans performs better because Iris clusters are approximately spherical.")

KMeans performs better because Iris clusters are approximately spherical.


Minimum Spanning Tree clustering connects all data points using the shortest possible total edge weight without forming cycles. It preserves the intrinsic geometric relationships between nearby points rather than relying on centroids. By removing the longest edges from the MST, natural separations between groups of points emerge as clusters. This approach allows detection of arbitrarily shaped clusters such as elongated or curved structures. Unlike K-Means clustering, MST clustering does not assume spherical cluster boundaries. It works effectively when clusters follow irregular manifolds in feature space. Since clustering decisions depend only on local distance structure, the method adapts well to non-linear patterns. Therefore, MST clustering is particularly useful for datasets where cluster shapes are complex or unevenly distributed.